# Lecture 3: Diffusion Models in Depth (Complete ELBO Derivation + NumPy Code)

<div class="nav-links">
  <a href="../../intro.html">About Me</a>
  <a href="../../publications.html">Publications</a>
  <a href="../../course_notes/my-teaching-philosophy.html">My Teaching</a>
  <a href="../../research_notes/intro.html">Research Notes</a>
</div>

---

This notebook provides a full variational derivation for diffusion models and then connects each equation to an executable minimal implementation.


## Learning Goals

1. Derive the forward process and all Gaussian identities used by DDPM training.
2. Derive the diffusion ELBO completely from first principles.
3. Derive why the practical training loss becomes a noise-prediction MSE.
4. Understand parameterization choices ($\\epsilon$, $x_0$, and $v$ prediction).
5. Implement and run a small NumPy denoiser with DDPM and DDIM-style samplers.


## 1. Setup and Notation

Let $x_0 \sim q_{\text{data}}(x_0)$ be a clean sample.

Define a forward noising Markov chain:

$$
q(x_t \mid x_{t-1}) = \mathcal N\left(\sqrt{\alpha_t}\,x_{t-1},\,\beta_t I\right),
\qquad \alpha_t = 1-\beta_t,
\qquad t=1,\dots,T.
$$

Define cumulative signal retention:

$$
\bar\alpha_t = \prod_{s=1}^{t} \alpha_s.
$$

We will repeatedly use this closed form:

$$
q(x_t\mid x_0) = \mathcal N\left(\sqrt{\bar\alpha_t}\,x_0,\,(1-\bar\alpha_t)I\right).
$$


### Derivation of $q(x_t\mid x_0)$

By recursion:

$$
x_t = \sqrt{\alpha_t}x_{t-1}+\sqrt{\beta_t}\,\epsilon_t,\quad \epsilon_t\sim\mathcal N(0,I).
$$

Unrolling to $x_0$ yields:

$$
x_t = \sqrt{\bar\alpha_t}x_0 + \sum_{s=1}^{t}
\left(\sqrt{\beta_s}\prod_{r=s+1}^{t}\sqrt{\alpha_r}\right)\epsilon_s.
$$

The sum of independent Gaussians is Gaussian, with variance:

$$
\sum_{s=1}^{t} \beta_s \prod_{r=s+1}^{t}\alpha_r = 1-\bar\alpha_t.
$$

Therefore:

$$
x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,
\quad \epsilon\sim\mathcal N(0,I).
$$


## 2. Exact Posterior $q(x_{t-1}\mid x_t, x_0)$

This posterior is needed inside ELBO KL terms.

Use Bayes (up to proportionality):

$$
q(x_{t-1}\mid x_t,x_0)
\propto
q(x_t\mid x_{t-1})q(x_{t-1}\mid x_0).
$$

Both factors are Gaussians in $x_{t-1}$, so their product is Gaussian.
Result:

$$
q(x_{t-1}\mid x_t,x_0)
=
\mathcal N\left(x_{t-1};\tilde\mu_t(x_t,x_0),\tilde\beta_t I\right),
$$

with

$$
\tilde\mu_t(x_t,x_0)
=
\frac{\sqrt{\bar\alpha_{t-1}}\beta_t}{1-\bar\alpha_t}x_0
+
\frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}x_t,
$$

$$
\tilde\beta_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t.
$$

This is the "true reverse" conditional when $x_0$ is known.


## 3. Reverse Generative Model

We learn:

$$
p_\theta(x_{0:T}) = p(x_T)\prod_{t=1}^{T} p_\theta(x_{t-1}\mid x_t),
\qquad p(x_T)=\mathcal N(0,I).
$$

Goal is maximum likelihood on data:

$$
\max_\theta \; \mathbb E_{q_{\text{data}}(x_0)}[\log p_\theta(x_0)].
$$

Direct optimization is intractable, so we optimize a variational bound.


## 4. Complete ELBO Derivation

Introduce forward process:

$$
q(x_{1:T}\mid x_0)=\prod_{t=1}^{T} q(x_t\mid x_{t-1}).
$$

Start from

$$
\log p_\theta(x_0) = \log \int p_\theta(x_{0:T})\,dx_{1:T}.
$$

Multiply/divide by $q(x_{1:T}\mid x_0)$:

$$
\log p_\theta(x_0)
=
\log\int q(x_{1:T}\mid x_0)
\frac{p_\theta(x_{0:T})}{q(x_{1:T}\mid x_0)}dx_{1:T}.
$$

Apply Jensen:

$$
\log p_\theta(x_0)
\ge
\mathbb E_{q(x_{1:T}\mid x_0)}
\left[
\log p_\theta(x_{0:T}) - \log q(x_{1:T}\mid x_0)
\right]
\equiv \mathcal L_{\text{ELBO}}(x_0).
$$

Minimize negative ELBO:

$$
\mathcal L_{\text{VLB}} = -\mathcal L_{\text{ELBO}}.
$$


### ELBO Decomposition into Standard DDPM Terms

Expand and regroup:

$$
\mathcal L_{\text{VLB}}
=
\underbrace{D_{\mathrm{KL}}\big(q(x_T\mid x_0)\|p(x_T)\big)}_{L_T}
+
\sum_{t=2}^{T}
\underbrace{\mathbb E_q\left[D_{\mathrm{KL}}\big(q(x_{t-1}\mid x_t,x_0)\|p_\theta(x_{t-1}\mid x_t)\big)\right]}_{L_{t-1}}
+
\underbrace{\mathbb E_q[-\log p_\theta(x_0\mid x_1)]}_{L_0}.
$$

Interpretation:

- $L_T$: terminal prior matching.
- $L_{t-1}$: reverse-step denoising fidelity.
- $L_0$: final reconstruction / discretization endpoint term.


## 5. From KL Terms to Practical MSE Objective

Parameterize reverse transition as Gaussian:

$$
p_\theta(x_{t-1}\mid x_t)=\mathcal N\big(x_{t-1};\mu_\theta(x_t,t),\sigma_t^2 I\big).
$$

Since $q(x_{t-1}\mid x_t,x_0)$ is Gaussian too, each KL has closed form and depends on squared mean mismatch:

$$
L_{t-1}
=
\mathbb E_q\left[
\frac{1}{2\sigma_t^2}\|\tilde\mu_t(x_t,x_0)-\mu_\theta(x_t,t)\|_2^2
\right]
+ C_t.
$$

The constants $C_t$ do not depend on $\theta$.


### $\epsilon$-Parameterization

Using

$$
x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,
$$

solve for $x_0$:

$$
x_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\epsilon}{\sqrt{\bar\alpha_t}}.
$$

Substitute into $\tilde\mu_t$ and simplify to obtain a linear form in $x_t$ and $\epsilon$.

Define network prediction $\epsilon_\theta(x_t,t)$ and set

$$
\mu_\theta(x_t,t)
=
\frac{1}{\sqrt{\alpha_t}}
\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon_\theta(x_t,t)\right).
$$

Then the weighted KL training objective becomes:

$$
\sum_{t=1}^{T} w_t\,\mathbb E_{x_0,\epsilon}\left[\|\epsilon-\epsilon_\theta(x_t,t)\|_2^2\right].
$$

In practice, DDPM often uses the simplified objective with uniform $t$ sampling and unit weights.


## 6. Alternative Targets: $x_0$-Prediction and $v$-Prediction

Equivalent parameterizations are commonly used:

- Predict clean sample $\hat x_0(x_t,t)$.
- Predict velocity-like target $v = \alpha_t \epsilon - \sigma_t x_0$ (exact form depends on schedule convention).

These are linear transforms of each other, but have different optimization/stability behavior under different noise schedules and guidance settings.


## 7. Training and Sampling Algorithms (High-Level)

**Training**

1. Sample $x_0$ from data.
2. Sample $t \sim \text{Uniform}\{1,\dots,T\}$.
3. Sample $\epsilon\sim\mathcal N(0,I)$ and construct $x_t$.
4. Predict $\epsilon_\theta(x_t,t)$.
5. Minimize MSE between predicted and true noise.

**Sampling (DDPM)**

1. Start from $x_T\sim\mathcal N(0,I)$.
2. For $t=T,\dots,1$, compute $\mu_\theta(x_t,t)$ and sample $x_{t-1}$.
3. Return $x_0$.

**Sampling (DDIM)**

Use deterministic update across a sparse timestep subset for fewer denoising evaluations.


In [ ]:
import numpy as np
import matplotlib
if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = matplotlib.rcParams.get
import matplotlib.pyplot as plt

plt.style.use('default')
np.set_printoptions(precision=4, suppress=True)


In [ ]:
# Schedule setup
T = 120
betas = np.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = np.cumprod(alphas)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(betas)
ax[0].set_title(r'Noise schedule $\beta_t$')
ax[0].set_xlabel('t')

ax[1].plot(alpha_bars)
ax[1].set_title(r'Cumulative signal $\bar\alpha_t$')
ax[1].set_xlabel('t')

plt.tight_layout()
plt.show()


In [ ]:
# Forward process demonstration on structured 2D data.
rng = np.random.default_rng(7)
n = 3000
ang = rng.uniform(0, 2 * np.pi, size=n)
rad = 2.0 + 0.55 * np.sin(3.5 * ang) + 0.14 * rng.normal(size=n)
x0 = np.column_stack([rad * np.cos(ang), rad * np.sin(ang)])

def q_sample(x0, t_idx, eps):
    return np.sqrt(alpha_bars[t_idx])[:, None] * x0 + np.sqrt(1 - alpha_bars[t_idx])[:, None] * eps

times = [0, 18, 45, 82, 119]
fig, ax = plt.subplots(1, len(times), figsize=(16, 3.2))
for a, t in zip(ax, times):
    eps = rng.normal(size=x0.shape)
    xt = q_sample(x0, np.full(len(x0), t), eps)
    a.scatter(xt[:, 0], xt[:, 1], s=4, alpha=0.24)
    a.set_title(f't={t+1}')
    a.axis('equal')
    a.set_xticks([])
    a.set_yticks([])
plt.suptitle('Forward diffusion: structure to noise', y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# Monte Carlo check of q(x_t|x_0) moments.
rng = np.random.default_rng(9)
t = 76
x0_scalar = 1.85
m = 160_000
eps = rng.normal(size=(m, 1))
xt = np.sqrt(alpha_bars[t]) * x0_scalar + np.sqrt(1 - alpha_bars[t]) * eps

print('Empirical mean :', float(xt.mean()))
print('Theory mean    :', float(np.sqrt(alpha_bars[t]) * x0_scalar))
print('Empirical var  :', float(xt.var()))
print('Theory var     :', float(1 - alpha_bars[t]))


## 8. Minimal NumPy Denoiser

The next helper cell defines a small MLP denoiser and sampling routines.
It is hidden so the derivation flow above stays readable.


In [ ]:
# Hidden helper cell: data, MLP denoiser, training loop, DDPM/DDIM samplers.

def make_target_data(n=9000, seed=0):
    rng = np.random.default_rng(seed)
    a = rng.uniform(0, 2 * np.pi, size=n)
    r = 2.15 + 0.4 * np.sin(4 * a) + 0.16 * rng.normal(size=n)
    X = np.column_stack([r * np.cos(a), r * np.sin(a)])
    X += 0.05 * rng.normal(size=X.shape)
    return X


def t_embed(t_idx, T):
    tn = (t_idx / (T - 1)).reshape(-1, 1)
    return np.concatenate([tn, np.sin(2 * np.pi * tn), np.cos(2 * np.pi * tn)], axis=1)


def make_inp(xt, t_idx, T):
    return np.concatenate([xt, t_embed(t_idx, T)], axis=1)


class TinyDenoiser:
    def __init__(self, in_dim=5, hidden=96, out_dim=2, seed=0):
        rng = np.random.default_rng(seed)
        self.p = {
            'W1': rng.normal(0, 0.07, size=(in_dim, hidden)),
            'b1': np.zeros(hidden),
            'W2': rng.normal(0, 0.07, size=(hidden, hidden)),
            'b2': np.zeros(hidden),
            'W3': rng.normal(0, 0.07, size=(hidden, out_dim)),
            'b3': np.zeros(out_dim),
        }

    def forward(self, inp):
        p = self.p
        a1 = inp @ p['W1'] + p['b1']
        h1 = np.tanh(a1)
        a2 = h1 @ p['W2'] + p['b2']
        h2 = np.tanh(a2)
        out = h2 @ p['W3'] + p['b3']
        return out, {'inp': inp, 'h1': h1, 'h2': h2}

    def backward(self, d_out, cache):
        p = self.p
        inp, h1, h2 = cache['inp'], cache['h1'], cache['h2']
        g = {}

        g['W3'] = h2.T @ d_out
        g['b3'] = d_out.sum(axis=0)

        dh2 = d_out @ p['W3'].T
        da2 = dh2 * (1 - h2 ** 2)
        g['W2'] = h1.T @ da2
        g['b2'] = da2.sum(axis=0)

        dh1 = da2 @ p['W2'].T
        da1 = dh1 * (1 - h1 ** 2)
        g['W1'] = inp.T @ da1
        g['b1'] = da1.sum(axis=0)
        return g

    def step(self, grads, lr):
        for k in self.p:
            self.p[k] -= lr * grads[k]

    def predict(self, inp):
        p = self.p
        h1 = np.tanh(inp @ p['W1'] + p['b1'])
        h2 = np.tanh(h1 @ p['W2'] + p['b2'])
        return h2 @ p['W3'] + p['b3']


def train_denoiser(model, data, *, steps=4200, batch_size=320, lr=1.8e-3, seed=0):
    rng = np.random.default_rng(seed)
    n = len(data)
    losses = []
    for _ in range(steps):
        idx = rng.integers(0, n, size=batch_size)
        x0 = data[idx]

        t_idx = rng.integers(0, T, size=batch_size)
        eps = rng.normal(size=x0.shape)
        xt = np.sqrt(alpha_bars[t_idx])[:, None] * x0 + np.sqrt(1 - alpha_bars[t_idx])[:, None] * eps

        inp = make_inp(xt, t_idx, T)
        pred, cache = model.forward(inp)
        diff = pred - eps
        loss = np.mean(diff ** 2)
        losses.append(loss)

        d_out = (2.0 / np.prod(diff.shape)) * diff
        grads = model.backward(d_out, cache)
        model.step(grads, lr)
    return losses


def ddpm_sample(model, n_samples=4500, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(n_samples, 2))
    for t in range(T - 1, -1, -1):
        t_arr = np.full(n_samples, t)
        eps_hat = model.predict(make_inp(x, t_arr, T))

        a = alphas[t]
        ab = alpha_bars[t]
        b = betas[t]
        mean = (x - (b / np.sqrt(1 - ab)) * eps_hat) / np.sqrt(a)

        if t > 0:
            ab_prev = alpha_bars[t - 1]
            var = b * (1 - ab_prev) / (1 - ab)
            x = mean + np.sqrt(var) * rng.normal(size=x.shape)
        else:
            x = mean
    return x


def ddim_sample(model, n_samples=4500, n_steps=24, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(n_samples, 2))
    ts = np.linspace(T - 1, 0, n_steps, dtype=int)

    for i, t in enumerate(ts):
        t_arr = np.full(n_samples, t)
        eps_hat = model.predict(make_inp(x, t_arr, T))
        ab_t = alpha_bars[t]

        x0_hat = (x - np.sqrt(1 - ab_t) * eps_hat) / np.sqrt(ab_t)
        if i == len(ts) - 1:
            x = x0_hat
        else:
            t_prev = ts[i + 1]
            ab_prev = alpha_bars[t_prev]
            x = np.sqrt(ab_prev) * x0_hat + np.sqrt(1 - ab_prev) * eps_hat
    return x


In [ ]:
Xtrain = make_target_data(n=9500, seed=13)
model = TinyDenoiser(in_dim=5, hidden=96, out_dim=2, seed=17)
losses = train_denoiser(model, Xtrain, steps=4600, batch_size=336, lr=1.7e-3, seed=21)

print(f"Final training MSE: {losses[-1]:.6f}")


In [ ]:
# Smoothed training curve
w = 120
sm = np.convolve(losses, np.ones(w) / w, mode='valid')

fig, ax = plt.subplots(figsize=(7.4, 3.8))
ax.plot(np.arange(w, len(losses) + 1), sm, color='#1f77b4')
ax.set_title('Noise-prediction training curve (smoothed)')
ax.set_xlabel('Step')
ax.set_ylabel('MSE')
plt.show()


In [ ]:
X_ddpm = ddpm_sample(model, n_samples=5000, seed=31)
X_ddim = ddim_sample(model, n_samples=5000, n_steps=26, seed=35)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
ax[0].scatter(Xtrain[:, 0], Xtrain[:, 1], s=4, alpha=0.22)
ax[0].set_title('Target data')

ax[1].scatter(X_ddpm[:, 0], X_ddpm[:, 1], s=4, alpha=0.22, color='#D62728')
ax[1].set_title('DDPM sampling')

ax[2].scatter(X_ddim[:, 0], X_ddim[:, 1], s=4, alpha=0.22, color='#2CA02C')
ax[2].set_title('DDIM-style few-step sampling')

for a in ax:
    a.axis('equal')
    a.set_xlabel('$x_1$')
    a.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()


## Summary

- We derived diffusion ELBO from first principles and decomposed it into standard DDPM terms.
- We derived the exact Gaussian posterior $q(x_{t-1}\mid x_t,x_0)$ and showed how KL terms reduce to denoising losses.
- We derived why $\epsilon$-prediction MSE is the practical objective.
- We verified key forward-process identities and trained a toy NumPy denoiser.

## Next

Continue to [Lecture 4: Latest Architectures in Diffusion](./lecture-4-latest-diffusion-architectures.ipynb).
